# FIT5196 Assessment 1 — Group030 Solution

**Group members:** King Man Chan · Yinglin Fang · Sizhe Hong · Xinhang Ren · Guohou Zhang

This notebook records how we inspected the two historical exports, cleaned and reconciled their records, and produced the six required relational tables. We kept the code in small steps so that each result can be checked before moving to the next step. Student IDs will be added before submission.

## Workflow roadmap

The JSON and XML sources contain overlapping business records, different nesting structures and inconsistent representations. The workflow follows: **inspect both sources → extract records at each entity grain → standardise comparable fields → reconcile records by stable primary key → construct six relational tables → validate their keys, relationships and calculations → export deterministic CSVs**.

## 0. Configuration and reproducibility

We define all file locations in one place and use relative paths throughout the notebook. The analysis runs offline and does not need an API or any downloaded resource.

In [1]:
from pathlib import Path
import json
import hashlib
import re
import sys
import xml.etree.ElementTree as ET
from collections import Counter
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'assignment_materials' / 'allocated_package').is_dir():
    repository_root = PROJECT_ROOT.parent.parent
    if (repository_root / 'assignment_materials' / 'allocated_package').is_dir():
        PROJECT_ROOT = repository_root

if (PROJECT_ROOT / 'assignment_materials' / 'allocated_package').is_dir():
    DATA_ROOT = PROJECT_ROOT / 'assignment_materials' / 'allocated_package'
    CODE_DIR = PROJECT_ROOT / 'processing' / 'code'
    OUTPUT_DIR = PROJECT_ROOT / 'processing' / 'outputs'
    TEMPLATE_DIR = PROJECT_ROOT / 'assignment_materials' / 'templates'
    MAPPING_PATH = PROJECT_ROOT / 'processing' / 'mapping' / 'Group030_source_to_target_mapping.csv'
else:
    DATA_ROOT = PROJECT_ROOT
    CODE_DIR = PROJECT_ROOT
    OUTPUT_DIR = PROJECT_ROOT / 'outputs'
    TEMPLATE_DIR = PROJECT_ROOT / 'templates'
    MAPPING_PATH = PROJECT_ROOT / 'Group030_source_to_target_mapping.csv'
sys.path.insert(0, str(CODE_DIR))
from Group030_text_functions import (
    MISSING, _remove_emoji, build_latin_analysis, clean_delivery_note, clean_narrative_text,
    contains_non_latin_script, extract_order_reference,
    extract_product_sku, extract_promo_code,
)

GROUP_ID = 'Group030'
INPUT_DIR = DATA_ROOT / 'raw_input'
DICTIONARY_PATH = DATA_ROOT / 'public_data_dictionary.csv'
MANIFEST_PATH = DATA_ROOT / 'A1_manifest.json'
TABLES = ['orders', 'order_items', 'customers', 'deliveries', 'products', 'product_reviews']

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

`GROUP_ID`, `INPUT_DIR`, `OUTPUT_DIR`, `TEMPLATE_DIR` and `DICTIONARY_PATH` are defined in the configuration cell. The raw files are only read; all generated files are written to `outputs`. `A1_manifest.json` is checked when available, but it is not required in the marking workspace.

### 0.1 Confirm the expected package files

We first check that the two source files and the supporting files used later in the notebook are present.

In [2]:
expected_files = {
    'dictionary': DICTIONARY_PATH,
    'JSON source': INPUT_DIR / f'{GROUP_ID}_commerce.json',
    'XML source': INPUT_DIR / f'{GROUP_ID}_operations.xml',
    'public text cases': TEMPLATE_DIR / 'A1_public_text_test_cases.csv',
}
file_presence = pd.DataFrame([
    {'role': role, 'path': path.relative_to(PROJECT_ROOT).as_posix(), 'exists': path.is_file(),
     'size_mb': round(path.stat().st_size / 1024**2, 3) if path.is_file() else None}
    for role, path in expected_files.items()
])
assert file_presence['exists'].all(), 'A required package file is missing'
file_presence

,role,path,exists,size_mb
0,dictionary,assignment_materials/allocated_package/public_...,True,0.011
1,JSON source,assignment_materials/allocated_package/raw_inp...,True,13.623
2,XML source,assignment_materials/allocated_package/raw_inp...,True,17.036
3,public text cases,assignment_materials/templates/A1_public_text_...,True,0.002


The folder contains one JSON commerce export and one XML operations export. Both files are small enough to parse in memory. The raw files are kept unchanged and are not included in the submission ZIP.

### 0.2 Verify group identity and source integrity

We use the manifest to confirm the group number, filenames, file sizes and SHA-256 hashes before processing the data.

In [3]:
if MANIFEST_PATH.is_file():
    with MANIFEST_PATH.open(encoding='utf-8') as handle:
        manifest = json.load(handle)
    assert manifest['group_alias'] == GROUP_ID
    manifest_identity = {
        'manifest_available': True,
        'manifest_group': manifest['group_alias'],
        'configured_group': GROUP_ID,
        'manifest_schema_version': manifest['manifest_schema_version'],
    }
else:
    manifest = None
    manifest_identity = {
        'manifest_available': False,
        'configured_group': GROUP_ID,
        'note': 'Optional manifest not supplied; required input presence is still checked above.',
    }
manifest_identity

{'manifest_available': True,
 'manifest_group': 'Group030',
 'configured_group': 'Group030',
 'manifest_schema_version': '1.1-public'}

In [4]:
if manifest is None:
    integrity = pd.DataFrame([{
        'status': 'SKIPPED',
        'detail': 'A1_manifest.json was not supplied; required input presence was checked separately.',
    }])
else:
    manifest_by_path = {entry['path']: entry for entry in manifest['files']}
    required_manifest_paths = [
        'public_data_dictionary.csv',
        f'raw_input/{GROUP_ID}_commerce.json',
        f'raw_input/{GROUP_ID}_operations.xml',
    ]
    integrity_rows = []
    for relative_path in required_manifest_paths:
        entry = manifest_by_path.get(relative_path)
        local_path = DATA_ROOT / relative_path
        actual_hash = hashlib.sha256(local_path.read_bytes()).hexdigest()
        integrity_rows.append({
            'manifest_path': relative_path,
            'local_path': local_path.relative_to(PROJECT_ROOT).as_posix(),
            'listed_in_manifest': entry is not None,
            'expected_bytes': entry['bytes'] if entry else None,
            'actual_bytes': local_path.stat().st_size,
            'sha256_match': entry is not None and actual_hash == entry['sha256'],
        })
    integrity = pd.DataFrame(integrity_rows)
    assert integrity['listed_in_manifest'].all(), 'A required input is absent from the manifest'
    assert integrity['sha256_match'].all(), 'Allocated input integrity check failed'
integrity

,manifest_path,local_path,listed_in_manifest,expected_bytes,actual_bytes,sha256_match
0,public_data_dictionary.csv,assignment_materials/allocated_package/public_...,True,11151,11151,True
1,raw_input/Group030_commerce.json,assignment_materials/allocated_package/raw_inp...,True,14284869,14284869,True
2,raw_input/Group030_operations.xml,assignment_materials/allocated_package/raw_inp...,True,17863595,17863595,True


The configured group is `Group030` and all hashes match the manifest. The original package README is stored locally as `README.source.md`; its contents have not been changed.

## 1. Parse and profile the two sources

We inspect the structure and values before applying any cleaning. The JSON file is read with `json.load` and the XML file with `xml.etree.ElementTree`. Regular expressions are used later only on narrative fields that have already been extracted.

### 1.1 Load the JSON export with a structured parser

In [5]:
json_path = INPUT_DIR / f'{GROUP_ID}_commerce.json'
with json_path.open(encoding='utf-8') as handle:
    json_data = json.load(handle)

print('Root Python type:', type(json_data).__name__)
print('Top-level keys:', list(json_data.keys()))

Root Python type: dict
Top-level keys: ['customerProfiles', 'exportMetadata', 'orders', 'productReviews']


The JSON root is a dictionary with four top-level objects: one metadata object and three collections of business records.

### 1.2 Measure each JSON top-level collection

A count at this stage is a raw source count, not an expected final answer. It is used later in row-flow reconciliation.

In [6]:
json_collections = []
for name, value in json_data.items():
    json_collections.append({
        'top_level_object': name,
        'python_type': type(value).__name__,
        'raw_elements': len(value) if isinstance(value, (list, dict)) else 1,
    })
json_collections = pd.DataFrame(json_collections)
json_collections

,top_level_object,python_type,raw_elements
0,customerProfiles,list,500
1,exportMetadata,dict,3
2,orders,list,2818
3,productReviews,list,3946


`customerProfiles`, `orders` and `productReviews` contain repeated records. `exportMetadata` describes the export itself and is not used as one of the six output entities. We retain these counts for the later row-flow checks.

### 1.3 Inspect JSON fields

We list the fields in a sample record from each collection. Full records are not printed because some review texts are very long.

In [7]:
json_field_profile = pd.DataFrame([
    {'collection': 'customerProfiles', 'fields': ' | '.join(json_data['customerProfiles'][0].keys())},
    {'collection': 'orders.header', 'fields': ' | '.join(json_data['orders'][0]['header'].keys())},
    {'collection': 'orders.shoppingCart[]', 'fields': ' | '.join(json_data['orders'][0]['shoppingCart'][0].keys())},
    {'collection': 'orders.delivery', 'fields': ' | '.join(json_data['orders'][0]['delivery'].keys())},
    {'collection': 'productReviews', 'fields': ' | '.join(json_data['productReviews'][0].keys())},
])
json_field_profile

,collection,fields
0,customerProfiles,accountStatus | acquisitionSource | ageBand | ...
1,orders.header,couponCode | couponDiscount | currency | custo...
2,orders.shoppingCart[],lineRevenue | orderID | orderItemID | productI...
3,orders.delivery,carrier | delayDays | delayReason | deliveredD...
4,productReviews,customerID | deliveryExperience | helpfulVotes...


A customer-profile element represents one customer. Each element in `orders` contains one header, a list of shopping-cart items and one delivery object. Each element in `productReviews` represents one review. From this structure, the candidate keys are `customerID`, `orderID`, `orderItemID`, `deliveryID` and `reviewID`; we check them in the next steps.

### 1.4 Profile JSON nesting and entity grains

The nested counts show how many records will enter each entity-specific staging collection before reconciliation.

In [8]:
json_nested_profile = {
    'orders': len(json_data['orders']),
    'order_items_nested_under_orders': sum(len(order['shoppingCart']) for order in json_data['orders']),
    'deliveries_nested_under_orders': sum(order.get('delivery') is not None for order in json_data['orders']),
    'reviews': len(json_data['productReviews']),
}
json_nested_profile

{'orders': 2818,
 'order_items_nested_under_orders': 8823,
 'deliveries_nested_under_orders': 2818,
 'reviews': 3946}

### 1.5 Check JSON candidate keys and duplicates

For each proposed entity key, we compare row count, unique count and missing count. This distinguishes entity identity from an ordinary attribute.

In [9]:
json_candidate_keys = {
    'customers.customerID': [x['customerID'] for x in json_data['customerProfiles']],
    'orders.orderID': [x['header']['orderID'] for x in json_data['orders']],
    'items.orderItemID': [item['orderItemID'] for order in json_data['orders'] for item in order['shoppingCart']],
    'deliveries.deliveryID': [x['delivery']['deliveryID'] for x in json_data['orders'] if x.get('delivery')],
    'reviews.reviewID': [x['reviewID'] for x in json_data['productReviews']],
}
json_key_profile = pd.DataFrame([
    {'candidate_key': name, 'rows': len(values), 'unique_values': len(set(values)),
     'duplicate_rows': len(values) - len(set(values)), 'missing_or_blank': sum(v is None or str(v).strip()=='' for v in values)}
    for name, values in json_candidate_keys.items()
])
json_key_profile

,candidate_key,rows,unique_values,duplicate_rows,missing_or_blank
0,customers.customerID,500,500,0,0
1,orders.orderID,2818,2750,68,0
2,items.orderItemID,8823,8612,211,0
3,deliveries.deliveryID,2818,2750,68,0
4,reviews.reviewID,3946,3850,96,0


Some candidate keys occur more than once in the source. We do not remove them at this point. The repeated records are first standardised and compared field by field; matching copies are then collapsed, while any difference is recorded as a conflict.

### 1.6 Profile JSON representation formats

In [10]:
json_header = json_data['orders'][0]['header']
json_format_examples = {
    'date': json_data['customerProfiles'][0]['signupDate'],
    'timestamp': json_header['orderTimestamp'],
    'boolean': json_header['expeditedDelivery'],
    'currency/number': json_header['deliveryCharges'],
    'percentage points': json_header['couponDiscount'],
    'optional string': repr(json_header['couponCode']),
    'narrative preview': json_header['customerNote'][:90],
}
json_format_examples

{'date': '2016-03-17',
 'timestamp': '2018-12-09 13:00:00',
 'boolean': True,
 'currency/number': 19.09,
 'percentage points': 5,
 'optional string': "''",
 'narrative preview': '[SYSTEM] <p>Please leave at reception</p> https://orders.example/help'}

The JSON file mainly uses ISO-style dates and timestamps, native booleans, and numeric money and discount values. Optional strings may be empty. Narrative fields contain examples of tags, markers, URLs and promotion references, which are handled in Section 3.

### 1.7 Load the XML export with a structured parser

In [11]:
xml_path = INPUT_DIR / f'{GROUP_ID}_operations.xml'
xml_tree = ET.parse(xml_path)
xml_root = xml_tree.getroot()

print('Root tag:', xml_root.tag)
print('Root attributes:', xml_root.attrib)

Root tag: OperationsExport
Root attributes: {'groupAlias': 'Group030', 'sourceSystem': 'OperationsERP', 'period': '2018'}


The XML root is `OperationsExport`. Its attributes identify `Group030`, the `OperationsERP` source system and the 2018 period.

### 1.8 Inspect XML top-level sections

In [12]:
xml_sections = pd.DataFrame([
    {'section': child.tag, 'direct_children': len(list(child))}
    for child in xml_root
])
xml_sections

,section,direct_children
0,Export_Metadata,2
1,Orders,2818
2,ProductCatalogue,1000
3,ProductReviews,3946
4,WarehouseDirectory,3


`Orders`, `ProductCatalogue` and `ProductReviews` contain records used in the six output tables. `WarehouseDirectory` provides warehouse reference information, while the required warehouse name is already present in each order header.

### 1.9 Count repeated XML elements at their natural grains

In [13]:
xml_element_profile = pd.DataFrame([
    {'structural_path': './Orders/Order', 'candidate_grain': 'one source order', 'raw_elements': len(xml_root.findall('./Orders/Order'))},
    {'structural_path': './Orders/Order/Shopping_Cart/Item', 'candidate_grain': 'one source order item', 'raw_elements': len(xml_root.findall('./Orders/Order/Shopping_Cart/Item'))},
    {'structural_path': './Orders/Order/Delivery', 'candidate_grain': 'one source delivery', 'raw_elements': len(xml_root.findall('./Orders/Order/Delivery'))},
    {'structural_path': './ProductCatalogue/Product', 'candidate_grain': 'one source product', 'raw_elements': len(xml_root.findall('./ProductCatalogue/Product'))},
    {'structural_path': './ProductReviews/Review', 'candidate_grain': 'one source review', 'raw_elements': len(xml_root.findall('./ProductReviews/Review'))},
])
xml_element_profile

,structural_path,candidate_grain,raw_elements
0,./Orders/Order,one source order,2818
1,./Orders/Order/Shopping_Cart/Item,one source order item,8885
2,./Orders/Order/Delivery,one source delivery,2818
3,./ProductCatalogue/Product,one source product,1000
4,./ProductReviews/Review,one source review,3946


### 1.10 Inspect one XML order's nested components

In [14]:
first_xml_order = xml_root.find('./Orders/Order')
xml_order_structure = pd.DataFrame([
    {'component': child.tag, 'repeated_children': len(list(child)),
     'child_tags': ' | '.join(grandchild.tag for grandchild in list(child)[:8])}
    for child in first_xml_order
])
xml_order_structure

,component,repeated_children,child_tags
0,Header,22,Order_ID | Source_System_Record_ID | Customer_...
1,Shopping_Cart,2,Item | Item
2,Delivery,20,Delivery_ID | Order_ID | Dispatch_Date | Promi...


Each XML order contains a `Header`, a `Shopping_Cart` and a `Delivery`. The shopping cart contains repeated `Item` elements, which are extracted separately at order-item grain.

### 1.11 Profile XML representation formats

In [15]:
xml_header = first_xml_order.find('Header')
xml_format_examples = {
    'date': first_xml_order.findtext('Delivery/Dispatch_Date'),
    'timestamp': xml_header.findtext('Order_Timestamp'),
    'boolean': xml_header.findtext('Expedited_Delivery'),
    'currency': xml_header.findtext('Order_Price'),
    'percentage': xml_header.findtext('Coupon_Discount'),
    'empty element': repr(xml_header.findtext('Coupon_Code')),
    'narrative preview': (xml_header.findtext('Customer_Note') or '')[:90],
}
xml_format_examples

{'date': '28/04/2018',
 'timestamp': '25/04/2018 09:33:00',
 'boolean': 'N',
 'currency': 'AUD 622.90',
 'percentage': '10%',
 'empty element': "''",
 'narrative preview': '[SYSTEM] <p>Call before delivery</p> https://orders.example/help'}

The XML file uses day-first dates and timestamps, `Y`/`N` booleans, money strings containing `AUD`, percentage strings containing `%`, and empty elements that parse as `None`. We standardise these representations before comparing XML records with JSON records.

### 1.12 Compare source coverage

In [16]:
source_coverage = pd.DataFrame([
    {'target_entity': 'customers', 'JSON': True, 'XML': False, 'source_structure': 'customerProfiles[]'},
    {'target_entity': 'orders', 'JSON': True, 'XML': True, 'source_structure': 'orders[].header / Orders/Order/Header'},
    {'target_entity': 'order_items', 'JSON': True, 'XML': True, 'source_structure': 'shoppingCart[] / Shopping_Cart/Item'},
    {'target_entity': 'deliveries', 'JSON': True, 'XML': True, 'source_structure': 'orders[].delivery / Order/Delivery'},
    {'target_entity': 'products', 'JSON': False, 'XML': True, 'source_structure': 'ProductCatalogue/Product'},
    {'target_entity': 'product_reviews', 'JSON': True, 'XML': True, 'source_structure': 'productReviews[] / ProductReviews/Review'},
])
source_coverage

,target_entity,JSON,XML,source_structure
0,customers,True,False,customerProfiles[]
1,orders,True,True,orders[].header / Orders/Order/Header
2,order_items,True,True,shoppingCart[] / Shopping_Cart/Item
3,deliveries,True,True,orders[].delivery / Order/Delivery
4,products,False,True,ProductCatalogue/Product
5,product_reviews,True,True,productReviews[] / ProductReviews/Review


Customers are supplied by JSON and products by XML. Orders, items, deliveries and reviews appear in both sources, so we check duplicates within each source and reconcile matching keys across the two sources. The next section records the source and processing rule for every required output field.

## 2. Source-to-target mapping

The mapping records how each required output field is produced. For every field, it lists the JSON and/or XML path, the conversion or calculation used, the treatment of overlapping records and the relevant notebook section. Its 111 rows correspond to the 111 required fields across the six output tables.

### 2.1 Load the completed mapping

The provided CSV contains the mapping IDs, output-table names and target-field names. We filled in the source paths and processing rules after inspecting both files.

In [17]:
mapping_path = MAPPING_PATH
mapping = pd.read_csv(mapping_path, keep_default_na=False)
print('Mapping rows:', len(mapping))
print('Mapping columns:', list(mapping.columns))
mapping.head(6)

Mapping rows: 111
Mapping columns: ['mapping_id', 'output_table', 'target_field', 'source_format', 'json_source_path', 'xml_source_path', 'transformation_or_derivation', 'overlap_or_conflict_rule', 'notebook_evidence']


,mapping_id,output_table,target_field,source_format,json_source_path,xml_source_path,transformation_or_derivation,overlap_or_conflict_rule,notebook_evidence
0,MAP-orders-01,orders,order_id,both,$.orders[].header.orderID,/OperationsExport/Orders/Order/Header/Order_ID,trim surrounding whitespace; preserve identifi...,normalise before comparing by table primary ke...,Section 4.8 orders; Section 5 reconciliation; ...
1,MAP-orders-02,orders,source_system_record_id,both,$.orders[].header.sourceSystemRecordID,/OperationsExport/Orders/Order/Header/Source_S...,trim surrounding whitespace; preserve identifi...,normalise before comparing by table primary ke...,Section 4.8 orders; Section 5 reconciliation; ...
2,MAP-orders-03,orders,customer_id,both,$.orders[].header.customerID,/OperationsExport/Orders/Order/Header/Customer_ID,trim surrounding whitespace; preserve identifi...,normalise before comparing by table primary ke...,Section 4.8 orders; Section 5 reconciliation; ...
3,MAP-orders-04,orders,order_timestamp,both,$.orders[].header.orderTimestamp,/OperationsExport/Orders/Order/Header/Order_Ti...,parse ISO JSON or day-first XML timestamp and ...,normalise before comparing by table primary ke...,Section 4.8 orders; Section 5 reconciliation; ...
4,MAP-orders-05,orders,sales_channel,both,$.orders[].header.salesChannel,/OperationsExport/Orders/Order/Header/Sales_Ch...,trim surrounding whitespace and preserve the s...,normalise before comparing by table primary ke...,Section 4.8 orders; Section 5 reconciliation; ...
5,MAP-orders-06,orders,payment_method,both,$.orders[].header.paymentMethod,/OperationsExport/Orders/Order/Header/Payment_...,trim surrounding whitespace and preserve the s...,normalise before comparing by table primary ke...,Section 4.8 orders; Section 5 reconciliation; ...


For example, `MAP-orders-04` describes how `orders.order_timestamp` is created. A mapping row therefore represents one output field rather than one business record.

### 2.2 Check mapping IDs and required entries

Each mapping ID must appear once. Every row must specify its source format, transformation, overlap/conflict treatment and notebook evidence. A JSON or XML path is blank only when that source does not supply the field.

In [18]:
mapping_core_columns = ['source_format','transformation_or_derivation','overlap_or_conflict_rule','notebook_evidence']
mapping_quality = {
    'rows': len(mapping),
    'unique_mapping_ids': mapping['mapping_id'].nunique(),
    'duplicate_mapping_ids': int(mapping['mapping_id'].duplicated().sum()),
    'rows_with_complete_core_entries': int(mapping[mapping_core_columns].ne('').all(axis=1).sum()),
    'allowed_source_formats_only': bool(mapping['source_format'].isin(['JSON','XML','both','derived']).all()),
}
assert mapping_quality['duplicate_mapping_ids'] == 0
assert mapping_quality['rows_with_complete_core_entries'] == len(mapping)
mapping_quality

{'rows': 111,
 'unique_mapping_ids': 111,
 'duplicate_mapping_ids': 0,
 'rows_with_complete_core_entries': 111,
 'allowed_source_formats_only': True}

All mapping IDs are unique and the four main description fields are complete. We next compare the target fields with the data dictionary and check the required source paths.

### 2.3 Compare mapping coverage with the public data dictionary

We compare `output_table + target_field` with the data dictionary to confirm that no required field is missing and no extra field has been added.

In [19]:
dictionary = pd.read_csv(DICTIONARY_PATH)
required_targets = set(zip(dictionary['output_table'], dictionary['field_name']))
mapped_targets = set(zip(mapping['output_table'], mapping['target_field']))
mapping_dictionary_check = {
    'dictionary_fields': len(required_targets),
    'mapped_fields': len(mapped_targets),
    'missing_from_mapping': sorted(required_targets - mapped_targets),
    'extra_in_mapping': sorted(mapped_targets - required_targets),
}
assert not mapping_dictionary_check['missing_from_mapping']
assert not mapping_dictionary_check['extra_in_mapping']
mapping_dictionary_check

{'dictionary_fields': 111,
 'mapped_fields': 111,
 'missing_from_mapping': [],
 'extra_in_mapping': []}

### 2.4 Summarise completeness by output table

In [20]:
mapping_by_table = (mapping.assign(core_complete=mapping[mapping_core_columns].ne('').all(axis=1))
    .groupby('output_table', as_index=False)
    .agg(required_fields=('mapping_id','size'), completed_fields=('core_complete','sum')))
mapping_by_table['completeness_pct'] = (100*mapping_by_table['completed_fields']/mapping_by_table['required_fields']).round(1)
mapping_by_table

,output_table,required_fields,completed_fields,completeness_pct
0,customers,20,20,100.0
1,deliveries,20,20,100.0
2,order_items,6,6,100.0
3,orders,23,23,100.0
4,product_reviews,21,21,100.0
5,products,21,21,100.0


All six output tables have complete mapping entries for their required fields.

### 2.5 Check source-path requirements

In [21]:
path_checks = pd.DataFrame({
    'mapping_id': mapping['mapping_id'], 'source_format': mapping['source_format'],
    'json_required': mapping['source_format'].isin(['JSON','both']),
    'json_present': mapping['json_source_path'].ne(''),
    'xml_required': mapping['source_format'].isin(['XML','both']),
    'xml_present': mapping['xml_source_path'].ne(''),
})
path_checks['path_rule_pass'] = ((~path_checks.json_required | path_checks.json_present) & (~path_checks.xml_required | path_checks.xml_present))
path_summary = path_checks.groupby('source_format').agg(rows=('mapping_id','size'), pass_rows=('path_rule_pass','sum'))
assert path_checks.path_rule_pass.all()
path_summary

,rows,pass_rows
source_format,,
JSON,20,20
XML,21,21
both,62,62
derived,8,8


### 2.6 Review representative mapping rules

The sample below shows the main types of processing used in the mapping: identifier preservation, timestamp parsing, currency conversion, arithmetic, narrative cleaning, multilingual processing and reference extraction. The separate CSV contains the full set of fields.

In [22]:
representative_mapping_ids = ['MAP-orders-01','MAP-orders-04','MAP-orders-11','MAP-orders-15','MAP-orders-22','MAP-order_items-06','MAP-products-21','MAP-product_reviews-11','MAP-product_reviews-17']
representative_mapping = mapping.loc[mapping.mapping_id.isin(representative_mapping_ids), ['mapping_id','output_table','target_field','source_format','json_source_path','xml_source_path','transformation_or_derivation']]
representative_mapping

,mapping_id,output_table,target_field,source_format,json_source_path,xml_source_path,transformation_or_derivation
0,MAP-orders-01,orders,order_id,both,$.orders[].header.orderID,/OperationsExport/Orders/Order/Header/Order_ID,trim surrounding whitespace; preserve identifi...
3,MAP-orders-04,orders,order_timestamp,both,$.orders[].header.orderTimestamp,/OperationsExport/Orders/Order/Header/Order_Ti...,parse ISO JSON or day-first XML timestamp and ...
10,MAP-orders-11,orders,delivery_charges,both,$.orders[].header.deliveryCharges,/OperationsExport/Orders/Order/Header/Delivery...,"remove AUD label and thousands separators, con..."
14,MAP-orders-15,orders,order_total,derived,derived order_price | $.orders[].header.coupon...,derived order_price | /OperationsExport/Orders...,calculate round(order_price * (1 - coupon_disc...
21,MAP-orders-22,orders,customer_note_clean,both,$.orders[].header.customerNote,/OperationsExport/Orders/Order/Header/Customer...,"after promo extraction, decode entities, apply..."
28,MAP-order_items-06,order_items,line_revenue,derived,$.orders[].shoppingCart[].quantity | $.orders[...,/OperationsExport/Orders/Order/Shopping_Cart/I...,"calculate round(quantity * unit_price, 2) for ..."
89,MAP-products-21,products,product_description_clean,XML,,/OperationsExport/ProductCatalogue/Product/Pro...,"decode entities, apply Unicode NFC, remove tag..."
100,MAP-product_reviews-11,product_reviews,review_body_latin_analysis,derived,derived product_reviews.review_body_clean,derived product_reviews.review_body_clean,derive from review_body_clean; retain Latin-sc...
106,MAP-product_reviews-17,product_reviews,extracted_order_reference,both,$.productReviews[].reviewText,/OperationsExport/ProductReviews/Review/Review...,extract bounded HORD/CORD plus exactly six dig...


Section 1 identified the available paths and source formats. The mapping records the rule for each target field. Sections 3–5 apply these rules, and Section 6 checks the resulting tables.

## 3. Clean and extract narrative text

The customer notes, delivery notes, product descriptions and reviews contain useful text together with tags, URLs, markers, emoji and embedded references. We first obtain each value from the parsed JSON object or XML element. The text functions are then applied to that value; they are not used to parse the document structure.

### Development note on AI assistance

We used conversational AI while planning this section to help organise the cleaning sequence and identify edge cases that could easily be missed, such as malformed references, mixed scripts and Latin letters with diacritics. We then checked each suggestion against the assignment specification and the supplied public cases. The final functions were reviewed against the raw examples and tested below; AI output was not treated as an expected answer.

### 3.1 Load the six required text functions

The functions are kept in `Group030_text_functions.py` because their names and arguments form part of the required test interface.

In [23]:
from Group030_text_functions import (
    _remove_emoji, clean_delivery_note, clean_narrative_text, extract_order_reference, extract_product_sku,
    extract_promo_code, build_latin_analysis, contains_non_latin_script,
)

required_text_functions = [
    clean_narrative_text, extract_order_reference, extract_product_sku,
    extract_promo_code, build_latin_analysis, contains_non_latin_script,
]
[(function.__name__, function.__doc__) for function in required_text_functions]

[('clean_narrative_text',
  "Accept None or a string; return lower-case cleaned text or 'NaN'."),
 ('extract_order_reference',
  "Accept None or a string; return the upper-case reference or 'NaN'."),
 ('extract_product_sku',
  "Accept None or a string; return the upper-case SKU or 'NaN'."),
 ('extract_promo_code',
  "Accept None or a string; return the upper-case code or 'NaN'."),
 ('build_latin_analysis',
  "Accept cleaned multilingual text; return Latin analysis or 'NaN'."),
 ('contains_non_latin_script',
  'Accept cleaned multilingual text; return a Python bool.')]

### 3.2 Check the narrative-cleaning sequence

For a narrative value, we extract any required business reference first. We then decode HTML entities, apply Unicode NFC, remove tags and the listed markers, remove URLs and emoji, remove complete reference/promotion wrappers, collapse whitespace and trim. Customer notes, product descriptions and review bodies are then lower-cased. Delivery notes follow the same cleaning sequence but keep the source letter case. The emoji step covers both single pictographs and common multi-code-point forms such as keycaps and joined family emoji. If no readable text remains, the function returns the literal string `NaN`.

In [24]:
cleaning_examples = pd.DataFrame({
    'raw_value': [
        '[SYSTEM] <p>Leave at reception</p> PROMO: B3SAVE-24 https://orders.example/help',
        '[SOURCE: mobile-app] <p>Caf&eacute; setup was easy 😊</p>',
        '[VERIFIED_PURCHASE] <div>Reliable     for daily use</div>',
        '',
    ]
})
cleaning_examples['cleaned_value'] = cleaning_examples['raw_value'].map(clean_narrative_text)
cleaning_examples

,raw_value,cleaned_value
0,[SYSTEM] <p>Leave at reception</p> PROMO: B3SA...,leave at reception
1,[SOURCE: mobile-app] <p>Caf&eacute; setup was ...,café setup was easy
2,[VERIFIED_PURCHASE] <div>Reliable for dail...,reliable for daily use
3,,NaN


The examples show that readable text is retained while the specified wrappers are removed. HTML entities are decoded before the final lowercase step, and an empty input produces the required text sentinel rather than a blank CSV field.

### 3.3 Extract order, SKU and promotion references

The three extraction functions use bounded patterns. A longer or embedded near-match is rejected instead of returning a valid-looking prefix.

In [25]:
reference_examples = pd.DataFrame({
    'raw_value': [
        'Reference: HORD123456 | SKU: SKU-ABC123',
        'PROMO: b3save-24',
        'XHORD1234567 is not a valid reference',
        'SKU-ABC123-extra is not accepted',
    ]
})
reference_examples['order_reference'] = reference_examples.raw_value.map(extract_order_reference)
reference_examples['product_sku'] = reference_examples.raw_value.map(extract_product_sku)
reference_examples['promo_code'] = reference_examples.raw_value.map(extract_promo_code)
reference_examples

,raw_value,order_reference,product_sku,promo_code
0,Reference: HORD123456 | SKU: SKU-ABC123,HORD123456,SKU-ABC123,NaN
1,PROMO: b3save-24,NaN,NaN,B3SAVE-24
2,XHORD1234567 is not a valid reference,NaN,NaN,NaN
3,SKU-ABC123-extra is not accepted,NaN,NaN,NaN


### 3.4 Preserve multilingual text and create the Latin analysis field

`review_body_clean` keeps valid letters from all scripts. `review_body_latin_analysis` is derived from that cleaned value and retains Latin letters, including European diacritics, while removing non-Latin letters. The separate boolean records whether the cleaned review contains a letter outside the Latin script.

In [26]:
multilingual_examples = pd.DataFrame({
    'review_body_clean': ['service était bon 包装很好', 'café setup was easy', '包装很好', 'NaN']
})
multilingual_examples['latin_analysis'] = multilingual_examples.review_body_clean.map(build_latin_analysis)
multilingual_examples['contains_non_latin'] = multilingual_examples.review_body_clean.map(contains_non_latin_script)
multilingual_examples

,review_body_clean,latin_analysis,contains_non_latin
0,service était bon 包装很好,service était bon,True
1,café setup was easy,café setup was easy,False
2,包装很好,NaN,True
3,NaN,NaN,False


The mixed-language example keeps the complete cleaned review but produces `service était bon` for Latin-only analysis. A fully non-Latin review is preserved in the multilingual field and returns `NaN` only in the Latin-analysis field.

### 3.5 Run the supplied public text cases

In [27]:
public_cases = pd.read_csv(TEMPLATE_DIR/'A1_public_text_test_cases.csv', keep_default_na=False)
public_results = []
for _, case in public_cases.iterrows():
    actual = str(globals()[case['function']](case['input_value']))
    public_results.append({
        'case_id': case['case_id'], 'function': case['function'],
        'expected': case['expected_output'], 'actual': actual,
        'status': 'PASS' if actual == case['expected_output'] else 'FAIL',
    })
public_results = pd.DataFrame(public_results)
public_results

,case_id,function,expected,actual,status
0,TXT-01,clean_narrative_text,leave at reception,leave at reception,PASS
1,TXT-02,extract_promo_code,B3SAVE-24,B3SAVE-24,PASS
2,TXT-03,clean_narrative_text,café setup was easy,café setup was easy,PASS
3,TXT-04,extract_order_reference,HORD123456,HORD123456,PASS
4,TXT-05,extract_product_sku,SKU-ABC123,SKU-ABC123,PASS
5,TXT-06,extract_order_reference,NaN,NaN,PASS
6,TXT-07,build_latin_analysis,service était bon,service était bon,PASS
7,TXT-08,contains_non_latin_script,True,True,PASS
8,TXT-09,build_latin_analysis,NaN,NaN,PASS
9,TXT-10,clean_narrative_text,reliable for daily use,reliable for daily use,PASS


In [28]:
public_test_summary = public_results['status'].value_counts().rename_axis('status').to_frame('cases')
assert public_results['status'].eq('PASS').all()
public_test_summary

,cases
status,
PASS,18


All 18 supplied cases pass. We load them with `keep_default_na=False` so the three characters `NaN` remain a string during comparison.

### 3.6 Run additional edge cases

We added cases for missing input, Latin diacritics, non-Latin detection and references next to extra identifier characters.

In [29]:
student_cases = [
    ('missing input', clean_narrative_text(None), 'NaN'),
    ('Latin diacritic', build_latin_analysis('déjà 東京'), 'déjà'),
    ('non-Latin flag', contains_non_latin_script('déjà 東京'), True),
    ('emoji variation', clean_narrative_text('notice ©️ ready'), 'notice ready'),
    ('keycap emoji', clean_narrative_text('press 1️⃣ now'), 'press now'),
    ('joined emoji', clean_narrative_text('family 👨‍👩‍👧‍👦 order'), 'family order'),
    ('flag emoji', clean_narrative_text('delivery 🇦🇺 local'), 'delivery local'),
    ('emoji-only input', clean_narrative_text('❤️'), 'NaN'),
    ('star emoji', clean_narrative_text('⭐ reliable product'), 'reliable product'),
    ('wrapper pipe', clean_narrative_text('Useful. Reference: HORD123456 | SKU: SKU-ABC123'), 'useful.'),
    ('wrapper slash', clean_narrative_text('Useful. Reference: HORD123456 / SKU: SKU-ABC123'), 'useful.'),
    ('wrapper semicolon', clean_narrative_text('Useful. Reference: HORD123456; SKU: SKU-ABC123'), 'useful.'),
    ('wrapper space', clean_narrative_text('Useful. Reference: HORD123456 SKU: SKU-ABC123'), 'useful.'),
    ('wrapper tab', clean_narrative_text('Useful. Reference: HORD123456\tSKU: SKU-ABC123'), 'useful.'),
    ('wrapper newline', clean_narrative_text('Useful. Reference: HORD123456\nSKU: SKU-ABC123'), 'useful.'),
    ('wrapper near match', clean_narrative_text('Reference: XHORD123456 / SKU: SKU-ABC123'), 'reference: xhord123456 / sku: sku-abc123'),
    ('delivery source case', clean_delivery_note('<p>Delivered Within Promise</p>'), 'Delivered Within Promise'),
    ('embedded order', extract_order_reference('XHORD123456'), 'NaN'),
    ('extended order prefix', extract_order_reference('x_HORD123456'), 'NaN'),
    ('extended order suffix', extract_order_reference('HORD123456-extra'), 'NaN'),
    ('extended SKU', extract_product_sku('SKU-ABC_extra'), 'NaN'),
    ('extended promo', extract_promo_code('B3SAVE-24_more'), 'NaN'),
]
student_results = pd.DataFrame(student_cases, columns=['case','actual','expected'])
student_results['status'] = student_results.apply(lambda row: 'PASS' if row.actual == row.expected else 'FAIL', axis=1)
assert student_results.status.eq('PASS').all()
student_results

,case,actual,expected,status
0,missing input,NaN,NaN,PASS
1,Latin diacritic,déjà,déjà,PASS
2,non-Latin flag,True,True,PASS
3,emoji variation,notice ready,notice ready,PASS
4,keycap emoji,press now,press now,PASS
5,joined emoji,family order,family order,PASS
6,flag emoji,delivery local,delivery local,PASS
7,emoji-only input,NaN,NaN,PASS
8,star emoji,reliable product,reliable product,PASS
9,wrapper pipe,useful.,useful.,PASS


## 4. Build the six standardised relational tables

We next convert the parsed source objects into records matching the data dictionary. Common conversion functions are defined once, followed by one normaliser for each shared entity type. Customers and products have a single source and are handled directly during table construction.

### 4.1 Standardise money and boolean values

In [30]:
def money(value):
    if isinstance(value, (int, float)): return float(value)
    return float(re.sub(r"[^0-9.+-]", "", str(value).replace(",", "")))

def boolean(value):
    if isinstance(value, bool): return value
    v = str(value).strip().lower()
    if v in {"true", "t", "yes", "y", "1"}: return True
    if v in {"false", "f", "no", "n", "0"}: return False
    raise ValueError(f"Unrecognised boolean: {value!r}")

`money` removes currency labels and thousands separators before numeric conversion. `boolean` accepts the JSON booleans and XML alternatives such as `Y` and `N`, and raises an error for an unknown value.

### 4.2 Standardise missing strings, dates and timestamps

In [31]:
def text(value):
    v = "" if value is None else str(value).strip()
    return v if v else MISSING

def date(value, dayfirst=False):
    return pd.to_datetime(value, dayfirst=dayfirst).strftime("%Y-%m-%d")

def timestamp(value, dayfirst=False):
    return pd.to_datetime(value, dayfirst=dayfirst).strftime("%Y-%m-%d %H:%M:%S")

def xml_record(element):
    return {child.tag: child.text for child in element}

Empty or missing string values become the literal string `NaN`. Dates are formatted as `YYYY-MM-DD`, while timestamps retain seconds as `YYYY-MM-DD HH:MM:SS`. The `dayfirst` argument is used for the XML formats observed in Section 1.

### 4.3 Define the order transformation

In [32]:
def normalise_order(h, source):
    get = (lambda a, b: h[a]) if source == "json" else (lambda a, b: h.findtext(b))
    raw_note = get("customerNote", "Customer_Note")
    discount_raw = get("couponDiscount", "Coupon_Discount")
    discount = float(discount_raw) if source == "json" else money(discount_raw)
    return {
        "order_id": text(get("orderID", "Order_ID")),
        "source_system_record_id": text(get("sourceSystemRecordID", "Source_System_Record_ID")),
        "customer_id": text(get("customerID", "Customer_ID")),
        "order_timestamp": timestamp(get("orderTimestamp", "Order_Timestamp"), source == "xml"),
        "sales_channel": text(get("salesChannel", "Sales_Channel")),
        "payment_method": text(get("paymentMethod", "Payment_Method")),
        "currency": text(get("currency", "Currency")),
        "nearest_warehouse": text(get("nearestWarehouse", "Nearest_Warehouse")),
        "order_status": text(get("orderStatus", "Order_Status")),
        "delivery_charges": round(money(get("deliveryCharges", "Delivery_Charges")), 2),
        "coupon_code": text(get("couponCode", "Coupon_Code")),
        "coupon_discount": discount,
        "season": text(get("season", "Season")),
        "expedited_delivery": boolean(get("expeditedDelivery", "Expedited_Delivery")),
        "customer_lat": float(get("customerLat", "Customer_Lat")),
        "customer_long": float(get("customerLong", "Customer_Long")),
        "device_type": text(get("deviceType", "Device_Type")),
        "referral_source": text(get("referralSource", "Referral_Source")),
        "customer_note_clean": clean_narrative_text(raw_note),
        "promo_code": extract_promo_code(raw_note),
    }

The order function maps differently named JSON/XML fields to one set of target names. It standardises the structured values, cleans the customer note and extracts a promotion code before the note wrapper is removed. The final monetary calculations are added after canonical order items have been reconciled.

### 4.4 Define the order-item transformation

In [33]:
def normalise_item(item, source):
    get = (lambda a, b: item[a]) if source == "json" else (lambda a, b: item.findtext(b))
    qty = int(get("quantity", "Quantity")); price = money(get("unitPrice", "Unit_Price"))
    return {"order_item_id": text(get("orderItemID", "Order_Item_ID")),
            "order_id": text(get("orderID", "Order_ID")), "product_id": text(get("productID", "Product_ID")),
            "quantity": qty, "unit_price": round(price, 2), "line_revenue": round(qty * price, 2)}

Each shopping-cart element becomes one order-item record. Quantity and unit price are converted before `line_revenue` is calculated and rounded to two decimal places.

### 4.5 Define the delivery transformation

In [34]:
def normalise_delivery(d, source):
    get = (lambda a, b: d[a]) if source == "json" else (lambda a, b: d.findtext(b))
    return {"delivery_id": text(get("deliveryID", "Delivery_ID")), "order_id": text(get("orderID", "Order_ID")),
      "dispatch_date": date(get("dispatchDate", "Dispatch_Date"), source == "xml"),
      "promised_date": date(get("promisedDate", "Promised_Date"), source == "xml"),
      "delivered_date": date(get("deliveredDate", "Delivered_Date"), source == "xml"),
      "carrier": text(get("carrier", "Carrier")), "service_level": text(get("serviceLevel", "Service_Level")),
      "delivery_status": text(get("deliveryStatus", "Delivery_Status")), "delay_days": int(get("delayDays", "Delay_Days")),
      "on_time_in_full": boolean(get("onTimeInFull", "On_Time_In_Full")),
      "fulfilment_hours": int(get("fulfilmentHours", "Fulfilment_Hours")),
      "delivery_cost": round(money(get("deliveryCost", "Delivery_Cost")), 2),
      "delay_reason": text(get("delayReason", "Delay_Reason")), "promised_days": int(get("promisedDays", "Promised_Days")),
      "tracking_event_count": int(get("trackingEventCount", "Tracking_Event_Count")),
      "delivery_window": text(get("deliveryWindow", "Delivery_Window")),
      "shipping_distance_km": float(get("shippingDistanceKm", "Shipping_Distance_Km")),
      "signature_required": boolean(get("signatureRequired", "Signature_Required")),
      "estimated_carbon_kg": float(get("estimatedCarbonKg", "Estimated_Carbon_Kg")),
      "delivery_note_clean": clean_delivery_note(get("deliveryNoteClean", "Delivery_Note_Clean"))}

Delivery dates, costs and flags are converted using the same rules for both sources. The three operational dates are retained separately so their order and the reported delay can be checked later.

### 4.6 Define the review transformation

In [35]:
def normalise_review(r, source):
    get = (lambda a, b: r[a]) if source == "json" else (lambda a, b: r.findtext(b))
    raw = get("reviewText", "Review_Text"); clean = clean_narrative_text(raw)
    return {"review_id": text(get("reviewID", "Review_ID")), "order_id": text(get("orderID", "Order_ID")),
      "order_item_id": text(get("orderItemID", "Order_Item_ID")), "product_id": text(get("productID", "Product_ID")),
      "customer_id": text(get("customerID", "Customer_ID")),
      "review_timestamp": timestamp(get("reviewTimestamp", "Review_Timestamp"), source == "xml"),
      "language_code": text(get("languageCode", "Language_Code")), "rating": int(get("rating", "Rating")),
      "review_title": text(get("reviewTitle", "Review_Title")), "review_body_clean": clean,
      "review_body_latin_analysis": build_latin_analysis(clean),
      "verified_purchase": boolean(get("verifiedPurchase", "Verified_Purchase")),
      "helpful_votes": int(get("helpfulVotes", "Helpful_Votes")),
      "review_length_chars": 0 if clean == MISSING else len(clean),
      "review_word_count": 0 if clean == MISSING else len(clean.split()),
      "contains_non_latin_script": contains_non_latin_script(clean),
      "extracted_order_reference": extract_order_reference(raw), "extracted_product_sku": extract_product_sku(raw),
      "delivery_experience": text(get("deliveryExperience", "Delivery_Experience")),
      "value_experience": text(get("valueExperience", "Value_Experience")),
      "writing_style": text(get("writingStyle", "Writing_Style"))}

The review function keeps its four relationship keys, standardises the timestamp and structured categories, and derives the two text fields, script flag, reference fields and review-length measures from the raw review text.

### 4.7 Reconcile records and construct the tables

In [36]:
def reconcile(records, key, table, conflicts):
    canonical = {}
    for source, row in records:
        k = row[key]
        if k not in canonical: canonical[k] = (row, {source})
        else:
            old, sources = canonical[k]
            differences = {}
            for field, new_value in row.items():
                old_value = old[field]
                old_missing = old_value is None or old_value == MISSING or pd.isna(old_value)
                new_missing = new_value is None or new_value == MISSING or pd.isna(new_value)
                if old_missing and not new_missing:
                    old[field] = new_value
                elif not old_missing and not new_missing and old_value != new_value:
                    differences[field] = (old_value, new_value)
            if differences:
                conflicts.append({"table": table, "key": k, "existing_sources": sorted(sources),
                                  "incoming_source": source, "differences": differences})
            sources.add(source)
    return [canonical[k][0] for k in sorted(canonical)]

Records with the same primary key are compared after standardisation. The first standardised row is retained only when repeated values agree; any field difference is added to the conflict list for validation.

In [37]:
def build_tables(input_dir, dictionary_path):
    with open(input_dir / f"{GROUP_ID}_commerce.json", encoding="utf-8") as f: js = json.load(f)
    root = ET.parse(input_dir / f"{GROUP_ID}_operations.xml").getroot()
    conflicts = []
    order_rows=[]; item_rows=[]; delivery_rows=[]; review_rows=[]
    for o in js["orders"]:
        order_rows.append(("JSON", normalise_order(o["header"], "json")))
        item_rows += [("JSON", normalise_item(x, "json")) for x in o["shoppingCart"]]
        if o.get("delivery"): delivery_rows.append(("JSON", normalise_delivery(o["delivery"], "json")))
    for o in root.findall("./Orders/Order"):
        order_rows.append(("XML", normalise_order(o.find("Header"), "xml")))
        item_rows += [("XML", normalise_item(x, "xml")) for x in o.findall("./Shopping_Cart/Item")]
        if o.find("Delivery") is not None: delivery_rows.append(("XML", normalise_delivery(o.find("Delivery"), "xml")))
    for r in js["productReviews"]: review_rows.append(("JSON", normalise_review(r, "json")))
    for r in root.findall("./ProductReviews/Review"): review_rows.append(("XML", normalise_review(r, "xml")))
    customer_rows=[]
    cmap={"customerID":"customer_id","signupDate":"signup_date","loyaltyTier":"loyalty_tier","customerSegment":"customer_segment","ageBand":"age_band","preferredChannel":"preferred_channel","homeSuburb":"home_suburb","prior12MOrders":"prior_12m_orders","lifetimeValueBeforePeriod":"lifetime_value_before_period","marketingConsent":"marketing_consent","homePostcode":"home_postcode","homeState":"home_state","homeCountry":"home_country","preferredLanguage":"preferred_language","acquisitionSource":"acquisition_source","accountStatus":"account_status","preferredDevice":"preferred_device","emailDomain":"email_domain","householdSizeBand":"household_size_band","contactFrequencyPreference":"contact_frequency_preference"}
    for x in js["customerProfiles"]:
        row={v:x[k] for k,v in cmap.items()}
        row["signup_date"]=date(row["signup_date"])
        row["prior_12m_orders"]=int(row["prior_12m_orders"])
        row["lifetime_value_before_period"]=round(money(row["lifetime_value_before_period"]),2)
        row["marketing_consent"]=boolean(row["marketing_consent"])
        for field in set(row)-{"prior_12m_orders","lifetime_value_before_period","marketing_consent"}:
            row[field]=text(row[field])
        customer_rows.append(("JSON",row))
    product_rows=[]
    pmap={"Product_ID":"product_id","Product_Name":"product_name","Category":"category","Brand":"brand","Unit_Price":"unit_price","Unit_Cost":"unit_cost","Launch_Year":"launch_year","Warranty_Months":"warranty_months","Weight_Kg":"weight_kg","Product_Sku":"product_sku","Subcategory":"subcategory","Model_Family":"model_family","Colour":"colour","Supplier_ID":"supplier_id","Supplier_Country":"supplier_country","Launch_Date":"launch_date","Tax_Category":"tax_category","Package_Type":"package_type","Recyclable_Packaging":"recyclable_packaging","Active_Flag":"active_flag","Product_Description":"product_description_clean"}
    for p in root.findall("./ProductCatalogue/Product"):
        x=xml_record(p); row={v:x.get(k) for k,v in pmap.items()}
        for f in ["unit_price","unit_cost"]: row[f]=round(money(row[f]),2)
        row["weight_kg"]=money(row["weight_kg"])
        for f in ["launch_year","warranty_months"]: row[f]=int(row[f])
        for f in ["recyclable_packaging","active_flag"]: row[f]=boolean(row[f])
        for f in set(row)-{"unit_price","unit_cost","weight_kg","launch_year","warranty_months","recyclable_packaging","active_flag","launch_date","product_description_clean"}:
            row[f]=text(row[f])
        row["launch_date"]=date(row["launch_date"], True)
        row["product_description_clean"]=clean_narrative_text(row["product_description_clean"])
        product_rows.append(("XML",row))
    tables={"orders":pd.DataFrame(reconcile(order_rows,"order_id","orders",conflicts)),
      "order_items":pd.DataFrame(reconcile(item_rows,"order_item_id","order_items",conflicts)),
      "customers":pd.DataFrame(reconcile(customer_rows,"customer_id","customers",conflicts)),
      "deliveries":pd.DataFrame(reconcile(delivery_rows,"delivery_id","deliveries",conflicts)),
      "products":pd.DataFrame(reconcile(product_rows,"product_id","products",conflicts)),
      "product_reviews":pd.DataFrame(reconcile(review_rows,"review_id","product_reviews",conflicts))}
    # Published arithmetic is derived from canonical item lines, not trusted source totals.
    sums=tables["order_items"].groupby("order_id",as_index=False).line_revenue.sum().rename(columns={"line_revenue":"order_price"})
    tables["orders"]=tables["orders"].merge(sums,on="order_id",validate="one_to_one")
    tables["orders"]["order_price"]=tables["orders"]["order_price"].round(2)
    tables["orders"]["tax_amount"]=(tables["orders"].order_price/11).round(2)
    tables["orders"]["order_total"]=(tables["orders"].order_price*(1-tables["orders"].coupon_discount/100)+tables["orders"].delivery_charges).round(2)
    dictionary=pd.read_csv(dictionary_path)
    for name,df in tables.items():
        cols=dictionary.loc[dictionary.output_table.eq(name)].sort_values("position").field_name.tolist()
        tables[name]=df[cols]
    source_key_sets={}
    for table, rows, key in [("orders",order_rows,"order_id"),("order_items",item_rows,"order_item_id"),("customers",customer_rows,"customer_id"),("deliveries",delivery_rows,"delivery_id"),("products",product_rows,"product_id"),("product_reviews",review_rows,"review_id")]:
        source_key_sets[table]={s:[row[key] for src,row in rows if src==s] for s in ["JSON","XML"]}
    duplicates={table:{src:len(keys)-len(set(keys)) for src,keys in sources.items()} for table,sources in source_key_sets.items()}
    overlap={table:len(set(sources["JSON"]) & set(sources["XML"])) for table,sources in source_key_sets.items()}
    key_flow={table:{"JSON rows":len(sources["JSON"]),"JSON unique":len(set(sources["JSON"])),
                     "XML rows":len(sources["XML"]),"XML unique":len(set(sources["XML"])),
                     "cross-source overlap":len(set(sources["JSON"]) & set(sources["XML"])),
                     "expected canonical":len(set(sources["JSON"]) | set(sources["XML"])),
                     "actual canonical":len(tables[table])}
              for table,sources in source_key_sets.items()}
    profile={"json":{"customers":len(js["customerProfiles"]),"orders":len(js["orders"]),
      "order_items":sum(len(x["shoppingCart"]) for x in js["orders"]),
      "deliveries":sum(bool(x.get("delivery")) for x in js["orders"]),"reviews":len(js["productReviews"])},
      "xml":{"orders":len(root.findall("./Orders/Order")),
      "order_items":len(root.findall("./Orders/Order/Shopping_Cart/Item")),
      "deliveries":len(root.findall("./Orders/Order/Delivery")),
      "products":len(root.findall("./ProductCatalogue/Product")),"reviews":len(root.findall("./ProductReviews/Review"))},
      "canonical":{k:len(v) for k,v in tables.items()}, "within_source_duplicates":duplicates,
      "cross_source_overlap":overlap, "key_flow":key_flow, "conflicts":conflicts}
    return tables, profile

`build_tables` reads the entity collections identified in Section 1, applies the relevant normaliser, reconciles shared entities and selects the fields in data-dictionary order. It also calculates order price, included GST and final order total from canonical order-item lines.

In [38]:
tables, profile = build_tables(INPUT_DIR, DICTIONARY_PATH)
row_counts = pd.DataFrame([
    {'table': name, 'rows': len(frame), 'columns': len(frame.columns)}
    for name, frame in tables.items()
])
row_counts

,table,rows,columns
0,orders,5000,23
1,order_items,15723,6
2,customers,500,20
3,deliveries,5000,20
4,products,1000,21
5,product_reviews,7000,21


### 4.8 Inspect the orders table

**Grain:** one canonical order. **Primary key:** `order_id`. Orders are supplied by both sources. The order amounts are recalculated from canonical item lines rather than copied from one source.

In [39]:
orders = tables['orders']
orders.head(3)

,order_id,source_system_record_id,customer_id,order_timestamp,sales_channel,payment_method,currency,nearest_warehouse,order_status,order_price,...,tax_amount,order_total,season,expedited_delivery,customer_lat,customer_long,device_type,referral_source,customer_note_clean,promo_code
0,HORD000001,SRC-030-H-000001,CUS00225,2018-09-07 10:05:00,Store,PayPal,AUD,Thompson,Completed,6916.92,...,628.81,6931.21,Spring,False,-37.882177,144.888263,Tablet,Social,no special instruction,NaN
1,HORD000002,SRC-030-H-000002,CUS00099,2018-02-23 21:51:00,Web,PayPal,AUD,Bakers,Completed,1867.88,...,169.81,1878.88,Summer,False,-37.819942,145.042315,Mobile,Social,gift purchase for a family member,NaN
2,HORD000003,SRC-030-H-000003,CUS00238,2018-08-05 13:53:00,Web,Card,AUD,Nickolson,Completed,7032.93,...,639.36,7061.68,Winter,True,-37.879295,144.975540,Tablet,Organic,call before delivery,NaN


In [40]:
orders_summary = {
    'rows': len(orders), 'unique_order_ids': orders.order_id.nunique(),
    'minimum_order_total': orders.order_total.min(),
    'maximum_order_total': orders.order_total.max(),
    'columns_in_dictionary_order': list(orders.columns),
}
orders_summary

{'rows': 5000,
 'unique_order_ids': 5000,
 'minimum_order_total': np.float64(26.86),
 'maximum_order_total': np.float64(15891.5),
 'columns_in_dictionary_order': ['order_id',
  'source_system_record_id',
  'customer_id',
  'order_timestamp',
  'sales_channel',
  'payment_method',
  'currency',
  'nearest_warehouse',
  'order_status',
  'order_price',
  'delivery_charges',
  'coupon_code',
  'coupon_discount',
  'tax_amount',
  'order_total',
  'season',
  'expedited_delivery',
  'customer_lat',
  'customer_long',
  'device_type',
  'referral_source',
  'customer_note_clean',
  'promo_code']}

### 4.9 Inspect the order-items table

**Grain:** one order item. **Primary key:** `order_item_id`. **Foreign keys:** `order_id` and `product_id`. Repeated shopping-cart arrays from each source are flattened at this grain.

In [41]:
order_items = tables['order_items']
order_items.head(3)

,order_item_id,order_id,product_id,quantity,unit_price,line_revenue
0,HITM0000001,HORD000001,PRD0054,2,2814.02,5628.04
1,HITM0000002,HORD000001,PRD0466,1,520.19,520.19
2,HITM0000003,HORD000001,PRD0628,1,62.19,62.19


In [42]:
order_items[['quantity','unit_price','line_revenue']].agg(['min','max','mean']).round(2)

,quantity,unit_price,line_revenue
min,1.00,15.23,15.23
max,3.00,4182.34,12547.02
mean,1.27,803.91,1025.10


### 4.10 Inspect the customers table

**Grain:** one customer. **Primary key:** `customer_id`. Customer profiles appear only in JSON. Postcodes are kept as strings, dates are standardised, and marketing consent is stored as a boolean.

In [43]:
customers = tables['customers']
customers.head(3)

,customer_id,signup_date,loyalty_tier,customer_segment,age_band,preferred_channel,home_suburb,prior_12m_orders,lifetime_value_before_period,marketing_consent,home_postcode,home_state,home_country,preferred_language,acquisition_source,account_status,preferred_device,email_domain,household_size_band,contact_frequency_preference
0,CUS00001,2016-03-17,Silver,Small Business,25-34,Store,Hawthorn,5,1876.75,False,3122,VIC,Australia,nl,Paid Search,Active,Mobile,example.net,2,Quarterly
1,CUS00002,2017-12-10,Bronze,Mainstream,45-54,Store,Hawthorn,14,5191.82,True,3122,VIC,Australia,de,Organic,Active,Mobile,inbox.example,3-4,Weekly
2,CUS00003,2014-09-05,Gold,Mainstream,35-44,Mobile,St Kilda,2,2135.34,True,3182,VIC,Australia,pl,Store,Active,Desktop,mail.test,5+,Monthly


In [44]:
customers[['loyalty_tier','customer_segment','preferred_language']].describe()

,loyalty_tier,customer_segment,preferred_language
count,500,500,500
unique,4,4,13
top,Bronze,Premium,ru
freq,190,151,49


### 4.11 Inspect the deliveries table

**Grain:** one completed order delivery. **Primary key:** `delivery_id`. **Foreign key:** `order_id`. Deliveries appear in both sources and are reconciled after date, money and boolean conversion.

In [45]:
deliveries = tables['deliveries']
deliveries.head(3)

,delivery_id,order_id,dispatch_date,promised_date,delivered_date,carrier,service_level,delivery_status,delay_days,on_time_in_full,fulfilment_hours,delivery_cost,delay_reason,promised_days,tracking_event_count,delivery_window,shipping_distance_km,signature_required,estimated_carbon_kg,delivery_note_clean
0,HDEL000001,HORD000001,2018-09-08,2018-09-13,2018-09-13,AusPost,Standard,Delivered,0,True,30,10.20,none,5,9,Afternoon,9.2947,True,0.943,Delivered within promise
1,HDEL000002,HORD000002,2018-02-24,2018-03-01,2018-02-27,Direct Freight,Standard,Delivered,0,True,19,7.67,none,5,8,Afternoon,4.2812,True,0.763,Delivered within promise
2,HDEL000003,HORD000003,2018-08-07,2018-08-12,2018-08-10,AusPost,Express,Delivered,0,True,48,21.04,none,5,6,Afternoon,6.7700,True,0.891,Delivered within promise


In [46]:
deliveries.groupby(['carrier','service_level']).size().rename('deliveries').reset_index()

,carrier,service_level,deliveries
0,AusPost,Express,205
1,AusPost,Standard,1050
2,DHL,Express,216
3,DHL,Standard,1029
4,Direct Freight,Express,254
5,Direct Freight,Standard,989
6,StarTrack,Express,253
7,StarTrack,Standard,1004


### 4.12 Inspect the products table

**Grain:** one product. **Primary key:** `product_id`. Products appear only in the XML catalogue. Price, cost, launch date, packaging flags and description text are standardised.

In [47]:
products = tables['products']
products.head(3)

,product_id,product_name,category,brand,unit_price,unit_cost,launch_year,warranty_months,weight_kg,product_sku,...,model_family,colour,supplier_id,supplier_country,launch_date,tax_category,package_type,recyclable_packaging,active_flag,product_description_clean
0,PRD0001,Candle Bloom 100,Laptop,Candle,2393.92,1475.97,2014,24,1.196,SKU-CAN00001,...,Arc,Silver,SUP001,Vietnam,2014-01-25,GST_STANDARD,Recycled box,False,True,ultrabook designed for portable document work ...
1,PRD0002,Vela Halo 101,Smartphone,Vela,1391.38,905.35,2012,24,0.261,SKU-VEL00002,...,Atlas,Green,SUP002,Malaysia,2012-08-14,GST_STANDARD,Protective case,True,True,5g smartphone designed for daily communication...
2,PRD0003,Candle Quest 102,Tablet,Candle,316.95,160.31,2017,24,0.610,SKU-CAN00003,...,Bloom,Graphite,SUP003,Korea,2017-10-07,GST_STANDARD,Recycled box,True,True,"productivity tablet designed for reading, anno..."


In [48]:
products.groupby('category').agg(products=('product_id','size'), mean_price=('unit_price','mean')).round(2)

,products,mean_price
category,,
Accessory,100,233.46
Audio,100,605.41
Gaming,100,1321.32
Home Entertainment,100,2077.14
Laptop,100,2059.45
Networking,100,804.95
Smart Home,100,424.41
Smartphone,100,1116.30
Tablet,100,918.22


### 4.13 Inspect the product-reviews table

**Grain:** one canonical product review. **Primary key:** `review_id`. Reviews are linked to orders, items, products and customers through four required foreign keys.

In [49]:
product_reviews = tables['product_reviews']
product_reviews.head(3)

,review_id,order_id,order_item_id,product_id,customer_id,review_timestamp,language_code,rating,review_title,review_body_clean,...,verified_purchase,helpful_votes,review_length_chars,review_word_count,contains_non_latin_script,extracted_order_reference,extracted_product_sku,delivery_experience,value_experience,writing_style
0,HREV000001,HORD000001,HITM0000001,PRD0054,CUS00225,2018-09-15 11:01:00,en,5,a pleasant shared-screen experience,vela spark 153 has become the screen we use mo...,...,True,47,1714,291,False,HORD000001,SKU-VEL00054,on_time,poor_value,comparison
1,HREV000002,HORD000001,HITM0000003,PRD0628,CUS00225,2018-10-28 12:02:00,en,5,helpful summaries after exercise,"i have used vela echo 727 after walks, short r...",...,True,40,1517,261,False,HORD000001,SKU-VEL00628,on_time,good_value,narrative
2,HREV000003,HORD000001,HITM0000004,PRD0620,CUS00225,2018-10-13 13:03:00,en,3,protective case that travels well,vela atlas 719 has carried my tablet and charg...,...,True,44,814,142,False,HORD000001,SKU-VEL00620,on_time,good_value,detailed


In [50]:
review_text_summary = product_reviews.groupby(['language_code','contains_non_latin_script']).agg(
    reviews=('review_id','size'), mean_rating=('rating','mean'), mean_clean_chars=('review_length_chars','mean')
).sort_values('reviews',ascending=False)
review_text_summary.head(12).round(2)

,,reviews,mean_rating,mean_clean_chars
language_code,contains_non_latin_script,,,
en,False,6291,3.71,1080.77
pl,False,73,4.19,1068.26
zh,True,72,3.35,408.56
ja,True,69,4.06,429.14
fr,False,64,3.31,1042.62
de,False,59,4.07,1029.68
ar,True,56,4.12,754.61
hi,True,56,3.57,800.93
nl,False,55,3.15,971.64


## 5. Reconcile duplicates and connect the six tables

Orders, order items, deliveries and reviews occur in both exports. We separate duplicates found inside one source from keys shared across the two sources, then check whether standardised overlapping records agree.

### 5.1 Count duplicate keys within each source

In [51]:
within_source_duplicates = pd.DataFrame(profile['within_source_duplicates']).T
within_source_duplicates.index.name = 'table'
within_source_duplicates

,JSON,XML
table,,
orders,68,68
order_items,211,217
customers,0,0
deliveries,68,68
products,0,0
product_reviews,96,96


The four shared entity types contain repeated primary keys within both exports. Customers and products are also included in the same check and have no within-source duplicates in this package. Every repeated row is compared through the same reconciliation function; no identifier list is hard-coded.

### 5.2 Count keys shared across JSON and XML

In [52]:
cross_source_overlap = pd.DataFrame([
    {'table': table, 'shared_keys': count}
    for table, count in profile['cross_source_overlap'].items()
])
cross_source_overlap

,table,shared_keys
0,orders,500
1,order_items,1557
2,customers,0
3,deliveries,500
4,products,0
5,product_reviews,700


A shared key represents the same business entity appearing in both systems. Orders, items, deliveries and reviews overlap across JSON and XML; customers are JSON-only and products are XML-only, so their cross-source overlap is zero by design.

### 5.3 Check field-level conflicts

In [53]:
conflict_summary = {
    'conflicting_keys': len(profile['conflicts']),
    'first_conflicts': profile['conflicts'][:5],
}
conflict_summary

{'conflicting_keys': 0, 'first_conflicts': []}

No standardised field conflicts were found. If a future input contains different non-missing values for the same key, the details will remain in this list rather than being hidden by a JSON-first or XML-first rule.

### 5.4 Check raw-to-canonical row flow

In [54]:
row_flow = pd.DataFrame(profile['key_flow']).T
row_flow.index.name = 'table'
row_flow['union_check'] = row_flow['expected canonical'].eq(row_flow['actual canonical'])
row_flow

,JSON rows,JSON unique,XML rows,XML unique,cross-source overlap,expected canonical,actual canonical,union_check
table,,,,,,,,
orders,2818,2750,2818,2750,500,5000,5000,True
order_items,8823,8612,8885,8668,1557,15723,15723,True
customers,500,500,0,0,0,500,500,True
deliveries,2818,2750,2818,2750,500,5000,5000,True
products,0,0,1000,1000,0,1000,1000,True
product_reviews,3946,3850,3946,3850,700,7000,7000,True


### 5.5 Record the required table relationships

In [55]:
relationships = pd.DataFrame([
    ('orders','customer_id','customers','customer_id'),
    ('order_items','order_id','orders','order_id'),
    ('order_items','product_id','products','product_id'),
    ('deliveries','order_id','orders','order_id'),
    ('product_reviews','order_id','orders','order_id'),
    ('product_reviews','order_item_id','order_items','order_item_id'),
    ('product_reviews','product_id','products','product_id'),
    ('product_reviews','customer_id','customers','customer_id'),
], columns=['child_table','foreign_key','parent_table','primary_key'])
relationships

,child_table,foreign_key,parent_table,primary_key
0,orders,customer_id,customers,customer_id
1,order_items,order_id,orders,order_id
2,order_items,product_id,products,product_id
3,deliveries,order_id,orders,order_id
4,product_reviews,order_id,orders,order_id
5,product_reviews,order_item_id,order_items,order_item_id
6,product_reviews,product_id,products,product_id
7,product_reviews,customer_id,customers,customer_id


These eight relationships are checked directly in the validation register. Keeping the tables at their own grains allows an order to retain several items and reviews without duplicating the order record itself.

## 6. Validate the standardised data

The validation register is produced from executable checks. Each row contains the check ID, what was checked, its observed result, status and our interpretation. Conversational AI was also used to help review the list of possible checks, but the final checks were selected from the assignment requirements and evaluated on our processed Group030 data.

In [56]:
def validate(tables, dictionary, profile):
    rows=[]
    def add(cid, check, passed, observed, resolution="None required"):
        rows.append({"validation_id":cid,"check":check,"status":"PASS" if passed else "FAIL","observed_result":str(observed),"resolution_or_interpretation":resolution})
    expected=set(TABLES); add("VAL-SCHEMA-01","All six required tables are present",set(tables)==expected,sorted(tables))
    for name,df in tables.items():
        exp=dictionary[dictionary.output_table.eq(name)].sort_values("position").field_name.tolist()
        add(f"VAL-SCHEMA-{TABLES.index(name)+2:02d}",f"{name} columns match dictionary order",list(df)==exp,f"{name}: {len(df)} rows, {len(df.columns)} ordered columns")
        pk={"orders":"order_id","order_items":"order_item_id","customers":"customer_id","deliveries":"delivery_id","products":"product_id","product_reviews":"review_id"}[name]
        add(f"VAL-PK-{TABLES.index(name)+1:02d}",f"{name} primary key is complete and unique",df[pk].notna().all() and df[pk].is_unique,f"{name}.{pk}: missing={df[pk].isna().sum()}, duplicates={df[pk].duplicated().sum()}")
        required=dictionary[(dictionary.output_table.eq(name)) & (~dictionary.nullable.astype(bool))].field_name
        blank=sum((df[f].astype(str).str.strip()=="").sum() for f in required)
        pandas_missing=sum(df[f].isna().sum() for f in required)
        allowed_literal_nan={"customer_note_clean","product_description_clean","delivery_note_clean",
                             "review_body_clean","review_body_latin_analysis","coupon_code","promo_code",
                             "extracted_order_reference","extracted_product_sku"}
        invalid_literal_nan=sum(df[f].astype(str).eq(MISSING).sum() for f in required if f not in allowed_literal_nan)
        missing_ok=blank==0 and pandas_missing==0 and invalid_literal_nan==0
        add(f"VAL-MISS-{TABLES.index(name)+1:02d}",f"{name} required and prescribed string values use the correct missing representation",missing_ok,
            f"empty={blank}, pandas missing={pandas_missing}, invalid literal NaN={invalid_literal_nan}")
        type_failures=[]
        for _,spec in dictionary[dictionary.output_table.eq(name)].iterrows():
            s=df[spec.field_name]; typ=spec.data_type
            if typ=="number": ok=pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s) and s.notna().all()
            elif typ=="boolean": ok=pd.api.types.is_bool_dtype(s) and s.notna().all()
            elif typ=="date": ok=s.astype(str).str.fullmatch(r"\d{4}-\d{2}-\d{2}").all() and pd.to_datetime(s,errors="coerce").notna().all()
            elif typ=="datetime": ok=s.astype(str).str.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}").all() and pd.to_datetime(s,errors="coerce").notna().all()
            else: ok=s.map(lambda value:isinstance(value,str)).all() and s.notna().all()
            if not ok:type_failures.append(spec.field_name)
        add(f"VAL-TYPE-{TABLES.index(name)+1:02d}",f"{name} fields conform to dictionary data types",not type_failures,f"type failures={type_failures}")
    fks=[("orders","customer_id","customers","customer_id"),("order_items","order_id","orders","order_id"),("order_items","product_id","products","product_id"),("deliveries","order_id","orders","order_id"),("product_reviews","order_id","orders","order_id"),("product_reviews","order_item_id","order_items","order_item_id"),("product_reviews","product_id","products","product_id"),("product_reviews","customer_id","customers","customer_id")]
    for i,(ct,cf,pt,pf) in enumerate(fks,1):
        missing=set(tables[ct][cf])-set(tables[pt][pf]); add(f"VAL-FK-{i:02d}",f"{ct}.{cf} references {pt}.{pf}",not missing,f"orphan keys={len(missing)}")
    duplicate_counts=profile["within_source_duplicates"]
    add("VAL-DUP-01","Within-source duplicate primary keys are counted for all six tables",set(duplicate_counts)==set(TABLES) and all(v>=0 for x in duplicate_counts.values() for v in x.values()),duplicate_counts,"Duplicate rows are compared field by field and collapsed by primary key")
    expected_shared={"orders","order_items","deliveries","product_reviews"}
    overlap_ok=all(profile["cross_source_overlap"][t]>0 for t in expected_shared) and all(profile["cross_source_overlap"][t]==0 for t in set(TABLES)-expected_shared)
    add("VAL-OVERLAP-01","Cross-source overlap is measured separately from within-source duplication",overlap_ok,profile["cross_source_overlap"])
    add("VAL-CONFLICT-01","Overlapping normalised records contain no field conflicts",not profile["conflicts"],f"normalised cross-source conflicts={len(profile['conflicts'])}","Investigate every listed field conflict before submission")
    for i,table in enumerate(TABLES,1):
        flow=profile["key_flow"][table]
        passed=flow["actual canonical"]==flow["expected canonical"]
        add(f"VAL-FLOW-{i:02d}",f"{table} canonical keys equal the union of unique JSON/XML keys",passed,flow)
    lines=tables["order_items"]
    line_calc=(lines.quantity*lines.unit_price).round(2)
    add("VAL-ARITH-00","Line revenue equals rounded quantity multiplied by unit price",(lines.line_revenue-line_calc).abs().le(.01).all(),f"max difference={(lines.line_revenue-line_calc).abs().max():.4f}")
    items=tables["order_items"].groupby("order_id").line_revenue.sum().round(2)
    actual=tables["orders"].set_index("order_id").order_price
    add("VAL-ARITH-01","Order price equals sum of rounded line revenues",(actual-items).abs().le(.01).all(),f"max difference={(actual-items).abs().max():.4f}")
    o=tables["orders"]
    calc=(o.order_price*(1-o.coupon_discount/100)+o.delivery_charges).round(2)
    add("VAL-ARITH-02","Order total applies discount then delivery without adding GST",(o.order_total-calc).abs().le(.01).all(),f"max difference={(o.order_total-calc).abs().max():.4f}")
    add("VAL-ARITH-03","Tax is included GST equal to order_price/11",(o.tax_amount-o.order_price.div(11).round(2)).abs().le(.01).all(),"GST not added to total")
    numeric_ok=(o.order_price.ge(0)&o.delivery_charges.ge(0)&o.coupon_discount.between(0,100)&o.customer_lat.between(-90,90)&o.customer_long.between(-180,180)).all()
    add("VAL-RANGE-01","Order numeric values fall in sensible ranges",numeric_ok,"non-negative money; discount 0–100; valid coordinates")
    add("VAL-RANGE-02","Item quantity is positive and price non-negative",tables["order_items"].quantity.gt(0).all() and tables["order_items"].unit_price.ge(0).all(),f"min quantity={tables['order_items'].quantity.min()}, min price={tables['order_items'].unit_price.min()}")
    add("VAL-RANGE-03","Review rating is 1–5 and helpful votes non-negative",tables["product_reviews"].rating.between(1,5).all() and tables["product_reviews"].helpful_votes.ge(0).all(),f"rating={tables['product_reviews'].rating.min()}–{tables['product_reviews'].rating.max()}")
    categorical={"sales_channel":{"Web","Store","Mobile"},"currency":{"AUD"},"order_status":{"Completed"},"service_level":{"Express","Standard"},"delivery_status":{"Delivered"},"rating":{1,2,3,4,5}}
    cat_bad={f:sorted(set((tables["orders"] if f in tables["orders"] else tables["deliveries"] if f in tables["deliveries"] else tables["product_reviews"])[f])-allowed) for f,allowed in categorical.items()}
    add("VAL-CAT-01","Published structured categories use observed allowed vocabularies",all(not x for x in cat_bad.values()),cat_bad)
    d=tables["deliveries"].merge(o[["order_id","order_timestamp"]],on="order_id")
    # Dispatch is date-only, so compare calendar dates (same-day dispatch is valid).
    temporal=(pd.to_datetime(d.order_timestamp).dt.normalize()<=pd.to_datetime(d.dispatch_date)) & (pd.to_datetime(d.dispatch_date)<=pd.to_datetime(d.delivered_date))
    add("VAL-TIME-01","Order date <= dispatch <= delivered",temporal.all(),f"violations={(~temporal).sum()}")
    promised=(pd.to_datetime(d.dispatch_date)<=pd.to_datetime(d.promised_date)); delivered=pd.to_datetime(d.delivered_date); promised_date=pd.to_datetime(d.promised_date); expected_delay=(delivered-promised_date).dt.days.clip(lower=0)
    add("VAL-TIME-02","Promised date is not before dispatch",promised.all(),f"violations={(~promised).sum()}")
    delay_consistency=d.delay_days.eq(expected_delay) & d.on_time_in_full.eq(delivered.le(promised_date))
    add("VAL-TIME-03","Delay days and OTIF agree with promised/delivered dates",delay_consistency.all(),f"violations={(~delay_consistency).sum()}")
    promised_days=(pd.to_datetime(d.promised_date)-pd.to_datetime(d.dispatch_date)).dt.days
    add("VAL-TIME-03B","Promised days agree with dispatch and promised dates",d.promised_days.eq(promised_days).all(),f"violations={(~d.promised_days.eq(promised_days)).sum()}")
    rv=tables["product_reviews"].merge(o[["order_id","order_timestamp"]],on="order_id")
    rt=pd.to_datetime(rv.review_timestamp)>=pd.to_datetime(rv.order_timestamp)
    add("VAL-TIME-04","Review timestamp is not before order timestamp",rt.all(),f"violations={(~rt).sum()}")
    sentinel_fields=[("orders","coupon_code"),("orders","promo_code"),("product_reviews","extracted_order_reference"),("product_reviews","extracted_product_sku"),("product_reviews","review_body_latin_analysis")]
    empty=sum((tables[t][f].astype(str).str.strip()=="").sum() for t,f in sentinel_fields)
    add("VAL-TEXT-01","Prescribed missing strings use literal NaN, not empty",empty==0,f"empty prescribed strings={empty}")
    nonlatin=tables["product_reviews"].contains_non_latin_script
    add("VAL-TEXT-02","Multilingual reviews and non-Latin indicators are preserved",nonlatin.any(),f"non-Latin reviews={nonlatin.sum()} of {len(nonlatin)}")
    refs=tables["product_reviews"]
    order_pattern=r"^(?:NaN|[HC]ORD\d{6})$"; sku_pattern=r"^(?:NaN|SKU-[A-Z0-9]+)$"
    add("VAL-TEXT-03","Extracted order references follow bounded format",refs.extracted_order_reference.str.fullmatch(order_pattern).all(),f"invalid formats={(~refs.extracted_order_reference.str.fullmatch(order_pattern)).sum()}")
    add("VAL-TEXT-04","Extracted SKUs follow bounded format",refs.extracted_product_sku.str.fullmatch(sku_pattern).all(),f"invalid formats={(~refs.extracted_product_sku.str.fullmatch(sku_pattern)).sum()}")
    promo=tables["orders"].promo_code
    promo_pattern=r"^(?:NaN|B[1-5]SAVE-\d{2})$"
    add("VAL-TEXT-04B","Extracted promotion codes follow bounded format",promo.str.fullmatch(promo_pattern).all(),f"invalid formats={(~promo.str.fullmatch(promo_pattern)).sum()}")
    review_lengths=refs.review_body_clean.map(lambda x:0 if x==MISSING else len(x))
    add("VAL-TEXT-05","Review character counts derive from cleaned multilingual text",review_lengths.eq(refs.review_length_chars).all(),f"mismatches={(review_lengths!=refs.review_length_chars).sum()}")
    review_words=refs.review_body_clean.map(lambda x:0 if x==MISSING else len(x.split()))
    add("VAL-TEXT-06","Review word counts derive from cleaned multilingual text",review_words.eq(refs.review_word_count).all(),f"mismatches={(review_words!=refs.review_word_count).sum()}")
    latin_expected=refs.review_body_clean.map(build_latin_analysis)
    nonlatin_expected=refs.review_body_clean.map(contains_non_latin_script)
    add("VAL-TEXT-07","Latin analysis is derived from cleaned multilingual review text",latin_expected.eq(refs.review_body_latin_analysis).all(),f"mismatches={(latin_expected!=refs.review_body_latin_analysis).sum()}")
    add("VAL-TEXT-08","Non-Latin indicator is derived from cleaned multilingual review text",nonlatin_expected.eq(refs.contains_non_latin_script).all(),f"mismatches={(nonlatin_expected!=refs.contains_non_latin_script).sum()}")
    lowercase_fields=[("orders","customer_note_clean"),("products","product_description_clean"),("product_reviews","review_body_clean")]
    lowercase_failures=sum((tables[t][f]!=MISSING).mul(tables[t][f].ne(tables[t][f].str.lower())).sum() for t,f in lowercase_fields)
    add("VAL-TEXT-09","The three designated cleaned narratives are lower-case",lowercase_failures==0,f"non-lower-case values={lowercase_failures}")
    delivery_notes=tables["deliveries"].delivery_note_clean
    uppercase_delivery_notes=delivery_notes.map(lambda value:value!=MISSING and any(char.isupper() for char in value)).sum()
    add("VAL-TEXT-10","Delivery notes retain source letter case instead of being forced to lower-case",uppercase_delivery_notes>0,f"delivery notes retaining upper-case letters={uppercase_delivery_notes}","Observed upper-case letters provide evidence that the delivery-specific cleaner preserves source case")
    wrapper_residue=refs.review_body_clean.str.contains(r"(?i)\breference\s*:|\bsku\s*:",regex=True).sum()+refs.review_body_latin_analysis.str.contains(r"(?i)\breference\s*:|\bsku\s*:",regex=True).sum()
    add("VAL-TEXT-11","Published review reference wrappers are absent from both cleaned review fields",wrapper_residue==0,f"wrapper residues={wrapper_residue}")
    emoji_residue=sum(_remove_emoji(value)!=value for field in ["review_body_clean","review_body_latin_analysis"] for value in refs[field])
    add("VAL-TEXT-12","No supported removable emoji remain in either cleaned review field",emoji_residue==0,f"emoji residues={emoji_residue}")
    return pd.DataFrame(rows)

In [57]:
dictionary = pd.read_csv(DICTIONARY_PATH)
validation_register = validate(tables, dictionary, profile)
validation_register.groupby(validation_register.validation_id.str.extract(r'VAL-([A-Z]+)', expand=False)).agg(
    checks=('validation_id','size'), passes=('status',lambda x:x.eq('PASS').sum()), failures=('status',lambda x:x.eq('FAIL').sum())
)

,checks,passes,failures
validation_id,,,
ARITH,4,4,0
CAT,1,1,0
CONFLICT,1,1,0
DUP,1,1,0
FK,8,8,0
FLOW,6,6,0
MISS,6,6,0
OVERLAP,1,1,0
PK,6,6,0


### 6.1 Schema, required values and data types

In [58]:
validation_register[validation_register.validation_id.str.startswith(('VAL-SCHEMA','VAL-MISS','VAL-TYPE'))]

,validation_id,check,status,observed_result,resolution_or_interpretation
0,VAL-SCHEMA-01,All six required tables are present,PASS,"['customers', 'deliveries', 'order_items', 'or...",None required
1,VAL-SCHEMA-02,orders columns match dictionary order,PASS,"orders: 5000 rows, 23 ordered columns",None required
3,VAL-MISS-01,orders required and prescribed string values u...,PASS,"empty=0, pandas missing=0, invalid literal NaN=0",None required
4,VAL-TYPE-01,orders fields conform to dictionary data types,PASS,type failures=[],None required
5,VAL-SCHEMA-03,order_items columns match dictionary order,PASS,"order_items: 15723 rows, 6 ordered columns",None required
7,VAL-MISS-02,order_items required and prescribed string val...,PASS,"empty=0, pandas missing=0, invalid literal NaN=0",None required
8,VAL-TYPE-02,order_items fields conform to dictionary data ...,PASS,type failures=[],None required
9,VAL-SCHEMA-04,customers columns match dictionary order,PASS,"customers: 500 rows, 20 ordered columns",None required
11,VAL-MISS-03,customers required and prescribed string value...,PASS,"empty=0, pandas missing=0, invalid literal NaN=0",None required
12,VAL-TYPE-03,customers fields conform to dictionary data types,PASS,type failures=[],None required


### 6.2 Primary and foreign keys

In [59]:
validation_register[validation_register.validation_id.str.startswith(('VAL-PK','VAL-FK'))]

,validation_id,check,status,observed_result,resolution_or_interpretation
2,VAL-PK-01,orders primary key is complete and unique,PASS,"orders.order_id: missing=0, duplicates=0",None required
6,VAL-PK-02,order_items primary key is complete and unique,PASS,"order_items.order_item_id: missing=0, duplicat...",None required
10,VAL-PK-03,customers primary key is complete and unique,PASS,"customers.customer_id: missing=0, duplicates=0",None required
14,VAL-PK-04,deliveries primary key is complete and unique,PASS,"deliveries.delivery_id: missing=0, duplicates=0",None required
18,VAL-PK-05,products primary key is complete and unique,PASS,"products.product_id: missing=0, duplicates=0",None required
22,VAL-PK-06,product_reviews primary key is complete and un...,PASS,"product_reviews.review_id: missing=0, duplicat...",None required
25,VAL-FK-01,orders.customer_id references customers.custom...,PASS,orphan keys=0,None required
26,VAL-FK-02,order_items.order_id references orders.order_id,PASS,orphan keys=0,None required
27,VAL-FK-03,order_items.product_id references products.pro...,PASS,orphan keys=0,None required
28,VAL-FK-04,deliveries.order_id references orders.order_id,PASS,orphan keys=0,None required


### 6.3 Duplicates, overlap, conflicts and row flow

In [60]:
validation_register[validation_register.validation_id.str.startswith(('VAL-DUP','VAL-OVERLAP','VAL-CONFLICT','VAL-FLOW'))]

,validation_id,check,status,observed_result,resolution_or_interpretation
33,VAL-DUP-01,Within-source duplicate primary keys are count...,PASS,"{'orders': {'JSON': 68, 'XML': 68}, 'order_ite...",Duplicate rows are compared field by field and...
34,VAL-OVERLAP-01,Cross-source overlap is measured separately fr...,PASS,"{'orders': 500, 'order_items': 1557, 'customer...",None required
35,VAL-CONFLICT-01,Overlapping normalised records contain no fiel...,PASS,normalised cross-source conflicts=0,Investigate every listed field conflict before...
36,VAL-FLOW-01,orders canonical keys equal the union of uniqu...,PASS,"{'JSON rows': 2818, 'JSON unique': 2750, 'XML ...",None required
37,VAL-FLOW-02,order_items canonical keys equal the union of ...,PASS,"{'JSON rows': 8823, 'JSON unique': 8612, 'XML ...",None required
38,VAL-FLOW-03,customers canonical keys equal the union of un...,PASS,"{'JSON rows': 500, 'JSON unique': 500, 'XML ro...",None required
39,VAL-FLOW-04,deliveries canonical keys equal the union of u...,PASS,"{'JSON rows': 2818, 'JSON unique': 2750, 'XML ...",None required
40,VAL-FLOW-05,products canonical keys equal the union of uni...,PASS,"{'JSON rows': 0, 'JSON unique': 0, 'XML rows':...",None required
41,VAL-FLOW-06,product_reviews canonical keys equal the union...,PASS,"{'JSON rows': 3946, 'JSON unique': 3850, 'XML ...",None required


### 6.4 Arithmetic, categories and numeric ranges

In [61]:
validation_register[validation_register.validation_id.str.startswith(('VAL-ARITH','VAL-CAT','VAL-RANGE'))]

,validation_id,check,status,observed_result,resolution_or_interpretation
42,VAL-ARITH-00,Line revenue equals rounded quantity multiplie...,PASS,max difference=0.0000,None required
43,VAL-ARITH-01,Order price equals sum of rounded line revenues,PASS,max difference=0.0000,None required
44,VAL-ARITH-02,Order total applies discount then delivery wit...,PASS,max difference=0.0000,None required
45,VAL-ARITH-03,Tax is included GST equal to order_price/11,PASS,GST not added to total,None required
46,VAL-RANGE-01,Order numeric values fall in sensible ranges,PASS,non-negative money; discount 0–100; valid coor...,None required
47,VAL-RANGE-02,Item quantity is positive and price non-negative,PASS,"min quantity=1, min price=15.23",None required
48,VAL-RANGE-03,Review rating is 1–5 and helpful votes non-neg...,PASS,rating=1–5,None required
49,VAL-CAT-01,Published structured categories use observed a...,PASS,"{'sales_channel': [], 'currency': [], 'order_s...",None required


### 6.5 Dates and operational consistency

In [62]:
validation_register[validation_register.validation_id.str.startswith('VAL-TIME')]

,validation_id,check,status,observed_result,resolution_or_interpretation
50,VAL-TIME-01,Order date <= dispatch <= delivered,PASS,violations=0,None required
51,VAL-TIME-02,Promised date is not before dispatch,PASS,violations=0,None required
52,VAL-TIME-03,Delay days and OTIF agree with promised/delive...,PASS,violations=0,None required
53,VAL-TIME-03B,Promised days agree with dispatch and promised...,PASS,violations=0,None required
54,VAL-TIME-04,Review timestamp is not before order timestamp,PASS,violations=0,None required


### 6.6 Text, references and multilingual fields

In [63]:
validation_register[validation_register.validation_id.str.startswith('VAL-TEXT')]

,validation_id,check,status,observed_result,resolution_or_interpretation
55,VAL-TEXT-01,"Prescribed missing strings use literal NaN, no...",PASS,empty prescribed strings=0,None required
56,VAL-TEXT-02,Multilingual reviews and non-Latin indicators ...,PASS,non-Latin reviews=303 of 7000,None required
57,VAL-TEXT-03,Extracted order references follow bounded format,PASS,invalid formats=0,None required
58,VAL-TEXT-04,Extracted SKUs follow bounded format,PASS,invalid formats=0,None required
59,VAL-TEXT-04B,Extracted promotion codes follow bounded format,PASS,invalid formats=0,None required
60,VAL-TEXT-05,Review character counts derive from cleaned mu...,PASS,mismatches=0,None required
61,VAL-TEXT-06,Review word counts derive from cleaned multili...,PASS,mismatches=0,None required
62,VAL-TEXT-07,Latin analysis is derived from cleaned multili...,PASS,mismatches=0,None required
63,VAL-TEXT-08,Non-Latin indicator is derived from cleaned mu...,PASS,mismatches=0,None required
64,VAL-TEXT-09,The three designated cleaned narratives are lo...,PASS,non-lower-case values=0,None required


### 6.7 Validation result

In [64]:
validation_status = validation_register.status.value_counts().to_dict()
assert validation_register.status.eq('PASS').all()
validation_status

{'PASS': 68}

All checks pass in this run. The register is also exported as a supporting CSV so individual checks can be referred to by their `VAL-...` IDs.

## 7. Export the six standardised tables

Only fields listed in the data dictionary are written to the submitted files. We then read each CSV back with `keep_default_na=False` to check the filenames, row counts, column order and literal `NaN` strings.

In [65]:
for table_name, frame in tables.items():
    output_path = OUTPUT_DIR / f'{GROUP_ID}_{table_name}_standardised.csv'
    frame.to_csv(output_path, index=False, na_rep='NaN')

validation_register.to_csv(OUTPUT_DIR/f'{GROUP_ID}_validation_register.csv', index=False)

In [66]:
export_check = []
for table_name, expected_frame in tables.items():
    output_path = OUTPUT_DIR/f'{GROUP_ID}_{table_name}_standardised.csv'
    reloaded = pd.read_csv(output_path, keep_default_na=False)
    export_check.append({
        'table': table_name, 'file': output_path.name, 'exists': output_path.is_file(),
        'rows': len(reloaded), 'columns': len(reloaded.columns),
        'column_order_matches': list(reloaded.columns) == list(expected_frame.columns),
        'size_kb': round(output_path.stat().st_size/1024,1),
    })
export_check = pd.DataFrame(export_check)
assert export_check.exists.all() and export_check.column_order_matches.all()
export_check

,table,file,exists,rows,columns,column_order_matches,size_kb
0,orders,Group030_orders_standardised.csv,True,5000,23,True,1045.7
1,order_items,Group030_order_items_standardised.csv,True,15723,6,True,726.7
2,customers,Group030_customers_standardised.csv,True,500,20,True,70.2
3,deliveries,Group030_deliveries_standardised.csv,True,5000,20,True,782.4
4,products,Group030_products_standardised.csv,True,1000,21,True,294.7
5,product_reviews,Group030_product_reviews_standardised.csv,True,7000,21,True,15973.1


## 8. Final run summary

The final checks below collect the main results from this run. They do not rely on manually entered expected row counts.

In [67]:
run_summary = {
    'group': GROUP_ID,
    'pandas_version': pd.__version__,
    'public_text_cases_passed': int(public_results.status.eq('PASS').sum()),
    'public_text_cases_total': len(public_results),
    'additional_text_cases_passed': int(student_results.status.eq('PASS').sum()),
    'validation_checks_passed': int(validation_register.status.eq('PASS').sum()),
    'validation_checks_total': len(validation_register),
    'six_output_files_present': len(list(OUTPUT_DIR.glob(f'{GROUP_ID}_*_standardised.csv'))) == 6,
}
assert run_summary['public_text_cases_passed'] == run_summary['public_text_cases_total']
assert run_summary['validation_checks_passed'] == run_summary['validation_checks_total']
assert run_summary['six_output_files_present']
run_summary

{'group': 'Group030',
 'pandas_version': '3.0.5',
 'public_text_cases_passed': 18,
 'public_text_cases_total': 18,
 'additional_text_cases_passed': 22,
 'validation_checks_passed': 68,
 'validation_checks_total': 68,
 'six_output_files_present': True}

The notebook completed from the configured raw files, recreated all six output tables and passed the text, relationship, arithmetic, temporal and export checks shown above.